<a href="https://colab.research.google.com/github/alieri72/automatizacoes-varias/blob/main/embeleza_tabelas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📊 Gerador Automático de Relatórios


Este programa transforma um ficheiro Excel com várias abas num relatório geral em HTML e em vários documentos Word (.docx) individuais, formatados de forma profissional.

Os ficheiros words ficam com os nomes das abas do excel.

---

📊 Como executar:


Clica no botão Play (▶️) no canto superior esquerdo da caixa de código logo abaixo.

Seleciona e carrega o teu ficheiro Excel quando o programa pedir.

Aguarda um instante: os relatórios finalizados serão descarregados automaticamente para o teu computador.

In [ ]:
# Instala automaticamente os pacotes necessários caso não estejam presentes
!pip install -q python-docx

import io
import zipfile
import pandas as pd
from google.colab import files
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# --- FUNÇÕES DE AUXÍLIO PARA O WORD ---
def set_cell_background(cell, fill_hex):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'), fill_hex)
    tcPr.append(shd)

def set_cell_margins(cell, top=130, bottom=130, left=160, right=160):
    tcPr = cell._tc.get_or_add_tcPr()
    tcMar = OxmlElement('w:tcMar')
    for margin_name, val in [('top', top), ('bottom', bottom), ('left', left), ('right', right)]:
        node = OxmlElement(f'w:{margin_name}')
        node.set(qn('w:w'), str(val))
        node.set(qn('w:type'), 'dxa')
        tcMar.append(node)
    tcPr.append(tcMar)

def set_cell_border(cell, color="CBD5E1", sz="4", val="single"):
    tcPr = cell._tc.get_or_add_tcPr()
    tcBorders = OxmlElement('w:tcBorders')
    for border_name in ['top', 'left', 'bottom', 'right']:
        border = OxmlElement(f'w:{border_name}')
        border.set(qn('w:val'), val)
        border.set(qn('w:sz'), sz)
        border.set(qn('w:space'), '0')
        border.set(qn('w:color'), color)
        tcBorders.append(border)
    tcPr.append(tcBorders)

# --- 1. UPLOAD INTERATIVO DO FICHEIRO EXCEL ---
print("Por favor, carregue o seu ficheiro Excel (ex: tabelas-todas.xlsx):")
uploaded = files.upload()

if not uploaded:
    print("❌ Nenhum ficheiro foi carregado.")
else:
    # Obter o nome do ficheiro carregado pelo utilizador
    nome_ficheiro_excel = list(uploaded.keys())[0]
    print(f"Ficheiro '{nome_ficheiro_excel}' carregado com sucesso!")

    # 2. Ler todas as folhas do Excel
    todas_as_folhas = pd.read_excel(nome_ficheiro_excel, sheet_name=None, dtype=str, parse_dates=False)

    html_tabelas_acumulado = ""
    documentos_docx = {}

    for nome_folha, df in todas_as_folhas.items():
        df = df.fillna("")

        # Tratamento rigoroso de datas literais
        if 'Data' in df.columns:
            def corrigir_data(val):
                val_str = str(val).strip()
                if '00:00:00' in val_str or hasattr(val, 'strftime'):
                    try:
                        ts = pd.to_datetime(val)
                        meses = {1: 'jan', 2: 'fev', 3: 'mar', 4: 'abr', 5: 'mai', 6: 'jun',
                                 7: 'jul', 8: 'ago', 9: 'set', 10: 'out', 11: 'nov', 12: 'dez'}
                        m = meses.get(ts.month, 'mes')
                        y = str(ts.year)[-2:]
                        return f"{m}/{y}"
                    except:
                        return val_str
                return val_str
            df['Data'] = df['Data'].apply(corrigir_data)

        # --- GERAÇÃO HTML ---
        styler = df.style.hide(axis="index").set_table_attributes('class="tabela-relatorio"').format(lambda val: str(val))

        for col in df.columns:
            col_lower = str(col).lower().strip()
            if any(k in col_lower for k in ['nº', 'id', 'num', 'data', 'fase', 'sessões']) or col_lower in ['', '0', '1']:
                styler = styler.set_properties(subset=[col], **{'text-align': 'center'})
            elif any(k in col_lower for k in ['valor', 'montante', 'quantia', 'saldo', '€']):
                styler = styler.set_properties(subset=[col], **{'text-align': 'right'})
            else:
                styler = styler.set_properties(subset=[col], **{'text-align': 'left'})

        tabela_html = styler.to_html(escape=False)
        html_tabelas_acumulado += f"""
        <div class="bloco-tabela">
            <h3>Tabela / Anexo: {nome_folha}</h3>
            {tabela_html}
        </div>
        <div class="quebra-pagina"></div>
        """

        # --- GERAÇÃO WORD (.docx) EM MEMÓRIA ---
        doc = Document()
        section = doc.sections[0]
        section.orientation = 1
        section.page_width = Inches(11.69)
        section.page_height = Inches(8.27)
        section.top_margin = section.bottom_margin = section.left_margin = section.right_margin = Inches(1.0)

        p_titulo = doc.add_paragraph()
        p_titulo.paragraph_format.space_after = Pt(12)
        run_titulo = p_titulo.add_run(f"Tabela / Anexo: {nome_folha}")
        run_titulo.font.name = 'Helvetica'
        run_titulo.font.size = Pt(14)
        run_titulo.font.bold = True
        run_titulo.font.color.rgb = RGBColor(0x1A, 0x36, 0x5D)

        tabela_docx = doc.add_table(rows=len(df) + 1, cols=len(df.columns))
        tabela_docx.alignment = WD_TABLE_ALIGNMENT.CENTER
        tabela_docx.autofit = False

        # Cabeçalho Word
        hdr_cells = tabela_docx.rows[0].cells
        for col_idx, coluna_nome in enumerate(df.columns):
            cell = hdr_cells[col_idx]
            cell.text = str(coluna_nome)
            set_cell_background(cell, "1A365D")
            set_cell_margins(cell, top=160, bottom=160, left=180, right=180)
            set_cell_border(cell, color="CBD5E1", sz="6")
            for p in cell.paragraphs:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                for run in p.runs:
                    run.font.name = 'Helvetica'
                    run.font.size = Pt(9)
                    run.font.bold = True
                    run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)

        trPr = tabela_docx.rows[0]._tr.get_or_add_trPr()
        trPr.append(OxmlElement('w:tblHeader'))

        # Linhas Word
        for row_idx, row_data in enumerate(df.itertuples(index=False), start=1):
            row_cells = tabela_docx.rows[row_idx].cells
            fundo_linha = "E2E8F0" if row_idx % 2 == 0 else "FFFFFF"
            for col_idx, valor in enumerate(row_data):
                cell = row_cells[col_idx]
                cell.text = str(valor)
                set_cell_background(cell, fundo_linha)
                set_cell_margins(cell, top=140, bottom=140, left=160, right=160)
                set_cell_border(cell, color="CBD5E1", sz="4")

                col_lower = str(df.columns[col_idx]).lower().strip()
                alinhamento = WD_ALIGN_PARAGRAPH.LEFT
                if any(k in col_lower for k in ['nº', 'id', 'num', 'data', 'fase', 'sessões']) or col_lower in ['', '0', '1']:
                    alinhamento = WD_ALIGN_PARAGRAPH.CENTER
                elif any(k in col_lower for k in ['valor', 'montante', 'quantia', 'saldo', '€']):
                    alinhamento = WD_ALIGN_PARAGRAPH.RIGHT

                for p in cell.paragraphs:
                    p.alignment = alinhamento
                    for run in p.runs:
                        run.font.name = 'Helvetica'
                        run.font.size = Pt(8)
                        run.font.color.rgb = RGBColor(0x2D, 0x37, 0x48)

        docx_io = io.BytesIO()
        doc.save(docx_io)
        docx_io.seek(0)
        nome_doc = f"tabela_{nome_folha.strip().lower().replace(' ', '_')}.docx"
        documentos_docx[nome_doc] = docx_io.getvalue()

    # --- MONTAGEM DO HTML FINAL ---
    estilos_html = """
    <style>
        * { -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }
        @page { size: A4 landscape; margin: 15mm; }
        body { font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; margin: 20px; color: #1A202C; }
        h2 { color: #1A365D; border-bottom: 2px solid #1A365D; padding-bottom: 8px; }
        h3 { color: #1A365D; margin-top: 30px; font-size: 14px; border-left: 4px solid #1A365D; padding-left: 8px; }
        table.tabela-relatorio { border-collapse: collapse; width: 100%; font-size: 11px; background-color: white; box-shadow: 0 4px 6px rgba(0,0,0,0.07); margin-bottom: 15px; }
        table.tabela-relatorio th { background-color: #1A365D !important; color: white !important; text-align: center; padding: 10px; border: 1px solid #CBD5E1; font-weight: bold; }
        table.tabela-relatorio td { padding: 9px 11px; border: 1px solid #CBD5E1; vertical-align: top; }
        table.tabela-relatorio tr:nth-child(even) { background-color: #E2E8F0 !important; }
        .quebra-pagina { page-break-after: always; }
    </style>
    """
    documento_html = f"<!DOCTYPE html><html lang='pt'><head><meta charset='UTF-8'><title>Relatório Consolidado</title>{estilos_html}</head><body><h2>Relatório Consolidado</h2>{html_tabelas_acumulado}</body></html>"

    nome_html_saida = "relatorio_consolidado.html"
    with open(nome_html_saida, "w", encoding="utf-8") as f:
        f.write(documento_html)

    # --- CRIAÇÃO DO ZIP COM OS WORD ---
    nome_zip_saida = "documentos_word_por_folha.zip"
    with zipfile.ZipFile(nome_zip_saida, "w") as zip_file:
        for nome_doc, conteudo in documentos_docx.items():
            zip_file.writestr(nome_doc, conteudo)

    # --- 3. DOWNLOAD AUTOMÁTICO PARA O COMPUTADOR DO UTILIZADOR ---
    print("\nProcessamento concluído! A transferir os ficheiros...")
    files.download(nome_html_saida)
    files.download(nome_zip_saida)
    print("Ficheiros descarregados com sucesso para a pasta de downloads do seu computador!")

Por favor, carregue o seu ficheiro Excel (ex: tabelas-todas.xlsx):


Saving zzz-tabelas-todas.xlsx to zzz-tabelas-todas.xlsx
Ficheiro 'zzz-tabelas-todas.xlsx' carregado com sucesso!

Processamento concluído! A transferir os ficheiros...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ficheiros descarregados com sucesso para a pasta de downloads do seu computador!
